# DMOSpeech 2: Input-Adaptive Handover Scheduling
## End-to-End Interactive Live Computational Tracer

**Author:** Rhenel Jhon Sajol (Master's Thesis 2026)  
**Method:** Dual-Mechanism Input-Adaptive Handover Scheduling (Prompt SNR Safety Gate + Zero-NFE Tangent Drift Probing)  
**Certified Decision Threshold:** $\hat{\lambda} = 0.50$ (PAC Bounded $P(R > 0.05) \le 0.10$)  
**Safety Gate Threshold:** $\theta = 20.0\text{ dB}$  

---

### What this Notebook Demonstrates:
This notebook provides a complete, step-by-step mathematical trace of the speech synthesis pipeline from reference audio and text to final waveform. At each phase, it prints the real numerical tensor norms, step calculations, tangent curvature drifts, and decision rules so you and your defense panel can inspect the exact computation end-to-end.

```
Reference Audio + Text ──────► [1. Prompt SNR Estimation]
                                       │
                       ┌───────────────┴───────────────┐
                       ▼                               ▼
             SNR < 20 dB (Noisy)              SNR >= 20 dB (Clean/Moderate)
             Safety Gate Fallback              Teacher Euler Sway Integration (t_0 -> t_8)
             Route to Step 8 (21 NFE)                  │
                       │                       Zero-NFE Tangent Drift: d_i
                       │                               │
                       │                       ┌───────┴───────┐
                       │                       ▼               ▼
                       │                 d_i <= 0.50       d_i > 0.50
                       │                 Early Handover    Continue Teacher
                       │                 (Step 4, 5, 6)    to Step 8
                       ▼                       ▼               ▼
                      Student Consistency Completion (j=1, 3 Steps)
                                       │
                            Vocos Vocoder (24 kHz)
                                       │
                               Synthesized Speech
```

--- 
### Step 0: Setup Environment & Load DMOSpeech 2 Models

In [ ]:
import os
import sys
import time
import math
import numpy as np
import soundfile as sf
import torch
import torchaudio
import matplotlib.pyplot as plt
import IPython.display as ipd
from torch.nn.utils.rnn import pad_sequence

# Add root and src to sys.path
NOTEBOOK_DIR = os.getcwd()
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(NOTEBOOK_DIR))) if "Handover Scheduling" in NOTEBOOK_DIR else NOTEBOOK_DIR
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
SRC_DIR = os.path.join(BASE_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
HANDOVER_DIR = os.path.join(BASE_DIR, "Handover Scheduling")
if HANDOVER_DIR not in sys.path:
    sys.path.insert(0, HANDOVER_DIR)

from infer import DMOInference
from f5_tts.infer.utils_infer import preprocess_ref_audio_text, convert_char_to_pinyin
from core.sampler import (
    SWAY_TEACHER_GRID,
    STUDENT_TAUS,
    STEP_METADATA,
    ALL_TARGET_STEPS,
    REFERENCE_STEP,
    estimate_prompt_snr,
    compute_stabilization_drift,
    make_local_generator,
    run_teacher_single_pass,
    run_student_completion,
    get_nfe,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STUDENT_CKPT = os.path.join(BASE_DIR, "ckpts", "model_85000.pt")
DP_CKPT = os.path.join(BASE_DIR, "ckpts", "model_1500.pt")

print(f"[Device] Using execution hardware: {DEVICE}")
print(f"[Model] Loading Teacher/Student Checkpoint: {STUDENT_CKPT}")
print(f"[Duration] Loading Duration Predictor Checkpoint: {DP_CKPT}")

tts = DMOInference(
    student_checkpoint_path=STUDENT_CKPT,
    duration_predictor_path=DP_CKPT,
    device=DEVICE,
    model_type="F5TTS_Base"
)
print("\n[OK] DMOSpeech 2 Engine Initialized Successfully!")

--- 
### Step 1: Interactive Inputs (Audio Reference + Target Text)

Enter your reference voice audio file, the transcript of what is spoken in that audio, and the new target text you want to synthesize.

> **Live Defense Toggle:** You can set `SIMULATE_NOISY_PROMPT = True` below to demonstrate the **Safety Gate** in real time! Adding +15 dB noise drops prompt SNR below 20 dB, immediately preventing catastrophic early handover.

In [ ]:
# ------------------------------------------------------------------
# Configure Your Demonstration Inputs
# ------------------------------------------------------------------
DEFAULT_WAV = os.path.join(BASE_DIR, "src", "f5_tts", "infer", "examples", "basic", "basic_ref_en.wav")

REF_AUDIO_PATH = DEFAULT_WAV  # Change to your own .wav path if testing custom voice
REF_TEXT = "Some call me nature, others call me mother nature."
TARGET_TEXT = "Input-adaptive teacher-to-student handover scheduling eliminates unnecessary compute while guaranteeing speech quality and acoustic stability."

# Defense Demonstration Toggles
SIMULATE_NOISY_PROMPT = False     # Set to True to test +15 dB prompt noise live!
CERTIFIED_LAMBDA = 0.50          # Prespecified PAC Certified Decision Threshold
SAFETY_GATE_THETA = 20.0         # Safety Gate Threshold in dB
BASE_SEED = 42

print("=" * 75)
print("CONFIGURED INPUTS:")
print(f"  Reference Audio : {REF_AUDIO_PATH}")
print(f"  Reference Text  : \"{REF_TEXT}\"")
print(f"  Target Text     : \"{TARGET_TEXT}\"")
print(f"  Simulate Noise  : {SIMULATE_NOISY_PROMPT}")
print("=" * 75)

# Play Reference Audio
print("\nReference Voice Playback:")
ipd.display(ipd.Audio(REF_AUDIO_PATH))

--- 
### Step 2: Audio Preprocessing & Prompt SNR Energy-Percentile VAD

The **Prompt SNR Safety Gate** evaluates whether the input speech prompt is clean enough to permit early handover. It computes the active speech power ($P_{\text{signal}}$, upper 50% energy frames) versus the noise floor ($P_{\text{noise}}$, lowest 10% energy frames):

$$\text{SNR} = 10 \cdot \log_{10} \left( \frac{P_{\text{signal}} + \epsilon}{P_{\text{noise}} + \epsilon} \right)$$

* **Rule:** If $\text{SNR} < 20.0\text{ dB}$, early handover is strictly prohibited and the pipeline is safely routed to the full 8-step teacher reference.

In [ ]:
# Load audio
waveform, sr = torchaudio.load(REF_AUDIO_PATH)
if sr != 24000:
    waveform = torchaudio.functional.resample(waveform, sr, 24000)
    sr = 24000
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)

# Optional: Inject calibrated 15 dB Gaussian noise if demonstrating noisy condition
if SIMULATE_NOISY_PROMPT:
    signal_pow = torch.mean(waveform ** 2)
    noise_pow = signal_pow / (10 ** (15.0 / 10.0))
    torch.manual_seed(BASE_SEED)
    noise = torch.randn_like(waveform) * torch.sqrt(noise_pow)
    waveform = waveform + noise
    print("[SIMULATION] Injected +15 dB Gaussian Noise into prompt waveform.")

wav_np = waveform.squeeze().cpu().numpy()
duration_sec = len(wav_np) / 24000.0

# Compute detailed energy breakdown for SNR estimation
frame_len = round(0.025 * 24000)  # 25 ms
hop_len = round(0.010 * 24000)    # 10 ms
num_frames = 1 + (len(wav_np) - frame_len) // hop_len
frames = np.lib.stride_tricks.as_strided(
    wav_np,
    shape=(num_frames, frame_len),
    strides=(wav_np.strides[0] * hop_len, wav_np.strides[0]),
)
energy = np.mean(frames**2, axis=-1)
sorted_energy = np.sort(energy)
noise_count = max(1, round(0.10 * len(sorted_energy)))
upper_start = len(sorted_energy) // 2
noise_energy = float(np.mean(sorted_energy[:noise_count]))
signal_energy = float(np.mean(sorted_energy[upper_start:]))
prompt_snr = estimate_prompt_snr(wav_np, sample_rate=24000)

print("=" * 75)
print("STEP 2: AUDIO FORENSIC & PROMPT SNR METRICS:")
print(f"  Waveform Samples : {len(wav_np):,} ({duration_sec:.2f} seconds)")
print(f"  Total Frames     : {num_frames} frames (25ms window, 10ms hop)")
print(f"  Active Speech P  : {signal_energy:.6e}")
print(f"  Noise Floor P    : {noise_energy:.6e}")
print(f"  Calculated SNR   : {prompt_snr:.2f} dB")
print(f"  Gate Threshold   : {SAFETY_GATE_THETA:.1f} dB")

if prompt_snr < SAFETY_GATE_THETA:
    gate_tripped = True
    print(f"  STATUS           : [ALERT] SAFETY GATE TRIPPED! SNR ({prompt_snr:.2f} dB) < {SAFETY_GATE_THETA:.1f} dB")
    print("                     -> Fallback to Step 8 (21 NFE) will be strictly enforced to prevent collapse.")
else:
    gate_tripped = False
    print(f"  STATUS           : [PASS] PROMPT IS CLEAN! SNR ({prompt_snr:.2f} dB) >= {SAFETY_GATE_THETA:.1f} dB")
    print("                     -> Permitted to probe trajectory curvature for early handover savings!")
print("=" * 75)

--- 
### Step 3: Text Tokenization, Mel Conditioning & Duration Prediction

The duration predictor calculates the target number of speech frames $T_{\text{target}}$ based on the text length and reference voice speaking rate.

In [ ]:
import tempfile

# Normalize audio RMS and resample
audio_tensor = torch.from_numpy(wav_np).float().unsqueeze(0)
rms = torch.sqrt(torch.mean(torch.square(audio_tensor)))
if rms < 0.3:
    audio_tensor = audio_tensor * 0.3 / rms
if sr != tts.target_sample_rate:
    audio_tensor = torchaudio.transforms.Resample(sr, tts.target_sample_rate)(audio_tensor)
audio_tensor = audio_tensor.to(DEVICE)

# Write temporary wav safely (closing file immediately to prevent Windows file locks)
temp_wav_path = os.path.join(tempfile.gettempdir(), f"tracer_prompt_{os.getpid()}.wav")
sf.write(temp_wav_path, wav_np, 24000)

ref_audio_clean, ref_text_clean = preprocess_ref_audio_text(temp_wav_path, REF_TEXT)

torch.manual_seed(BASE_SEED)
predicted_dur = tts.predict_duration(
    temp_wav_path,
    TARGET_TEXT,
    REF_TEXT,
    dp_softmax_range=0.7,
    temperature=0,
)
target_dur = max(10, int(predicted_dur))
if os.path.exists(temp_wav_path):
    try:
        os.remove(temp_wav_path)
    except OSError:
        pass

# Text phonetic preparation
text_list = [ref_text_clean + TARGET_TEXT]
final_text_list = convert_char_to_pinyin(text_list)

ref_audio_len = audio_tensor.shape[-1] // tts.hop_length
total_duration = ref_audio_len + target_dur

# Prepare input representations
cond_tensor, text_ids, step_cond, cond_mask, max_dur, dur_tensor = tts._prepare_inputs(
    audio_tensor, final_text_list, total_duration
)
prompt_len = cond_mask.sum().item()

# Draw initial Gaussian noise y0 with deterministic local generator
gen = make_local_generator(BASE_SEED)
y0 = [torch.randn(dur, 100, dtype=torch.float32, generator=gen) for dur in dur_tensor]
y0 = pad_sequence(y0, padding_value=0, batch_first=True).to(DEVICE)

print("=" * 75)
print("STEP 3: LATENT CONDITIONING & DURATION BREAKDOWN:")
print(f"  Reference Mel Frames (Prompt) : {prompt_len} frames ({prompt_len * 256 / 24000:.2f}s)")
print(f"  Target Mel Frames (Synthesis) : {target_dur} frames ({target_dur * 256 / 24000:.2f}s)")
print(f"  Total Integration Horizon     : {total_duration} frames ({total_duration * 256 / 24000:.2f}s)")
print(f"  Conditioning Tensor Shape     : {list(step_cond.shape)}")
print(f"  Initial Noise Tensor Norm ||y0||: {torch.norm(y0):.4f}")
print("=" * 75)

--- 
### Step 4: Teacher Integration & Zero-Overhead Trajectory Drift Probing ($t_0 \to t_8$)

At each teacher step $i$ along the sway grid $t_i = 1 - \cos(\frac{\pi}{2} \cdot \frac{i}{16})$:
1. **Guided Velocity Evaluation:** Computes $v_i = v_\theta(x_{t_i}, t_i)$ using Classifier-Free Guidance (CFG factor = 2.0).
2. **Projected Endpoint Estimation:** Computes the projected clean mel estimate $\hat{x}_1(t_i)$ as an algebraic byproduct with **zero neural forward pass overhead**:
   $$\hat{x}_1(t_i) = x_{t_i} + (1 - t_i) \cdot v_i$$
3. **Zero-NFE Tangent Drift Calculation:** Measures the trajectory displacement relative to endpoint magnitude on the generated frames $[T_{\text{prompt}}:]$:
   $$d_i = \frac{\|\hat{x}_1(t_i) - \hat{x}_1(t_{i-1})\|_2}{\|\hat{x}_1(t_i)\|_2 + \epsilon}$$
4. **Certified PAC Decision:** If $d_i \le \hat{\lambda} = 0.50$, early handover triggers immediately!

In [ ]:
print("=" * 105)
print(f"{'Step':<5} | {'t_curr':<8} | {'t_next':<8} | {'||v_i||_2':<12} | {'||x_1(t_i)||_2':<15} | {'Drift d_i':<11} | {'d_i <= 0.50?':<14} | {'Policy Decision':<20}")
print("-" * 105)

state = y0.clone()
prev_endpoint = None
endpoints = {}
drift_trace = {}
chosen_step = None
start_teacher_time = time.time()

with torch.inference_mode():
    for i in range(9):  # Steps 0 through 8
        t_curr_val = SWAY_TEACHER_GRID[i]
        t_next_val = SWAY_TEACHER_GRID[i + 1]
        t_curr = torch.tensor(t_curr_val, device=DEVICE, dtype=torch.float32)

        # Two forward evaluations for Classifier-Free Guidance (CFG)
        pred = tts.teacher(
            x=state, cond=step_cond, text=text_ids, time=t_curr,
            mask=None, drop_audio_cond=False, drop_text=False
        )
        null_pred = tts.teacher(
            x=state, cond=step_cond, text=text_ids, time=t_curr,
            mask=None, drop_audio_cond=True, drop_text=True
        )
        v_curr = pred + (pred - null_pred) * 2.0  # CFG = 2.0
        v_norm = float(torch.norm(v_curr).item())

        # Zero-NFE Projected Endpoint Calculation
        endpoint_curr = state + (1.0 - t_curr_val) * v_curr
        ep_norm = float(torch.norm(endpoint_curr[:, prompt_len:, :]).item())

        # Calculate Tangent Drift on generated speech frames
        d_i_str = "--"
        is_stable_str = "--"
        decision_str = "Integrating..."

        if prev_endpoint is not None:
            d_i = compute_stabilization_drift(endpoint_curr, prev_endpoint, prompt_len)
            drift_trace[i] = d_i
            d_i_str = f"{d_i:.4f}"

            if i in (4, 5, 6):
                is_stable = (d_i <= CERTIFIED_LAMBDA)
                is_stable_str = "YES (<= 0.50)" if is_stable else "NO (> 0.50)"
                if is_stable and not gate_tripped and chosen_step is None:
                    chosen_step = i
                    decision_str = f"[CHOSEN] EXIT AT STEP {i}!"
                elif gate_tripped:
                    decision_str = "Gate Tripped (Hold)"
                else:
                    decision_str = "Drift High (Continue)"
            elif i == 8:
                decision_str = "Reference Horizon"

        endpoints[i] = endpoint_curr.clone()

        print(f"{i:<5} | {t_curr_val:<8.4f} | {t_next_val:<8.4f} | {v_norm:<12.4f} | {ep_norm:<15.4f} | {d_i_str:<11} | {is_stable_str:<14} | {decision_str:<20}")

        # Euler step to next state
        if i < 8:
            dt = t_next_val - t_curr_val
            state = state + dt * v_curr
            prev_endpoint = endpoint_curr

teacher_elapsed = time.time() - start_teacher_time
print("-" * 105)

# Final policy resolution
if chosen_step is None:
    chosen_step = 8
    policy_status = "Step 8 Reference Fallback (Safety Gate Active or Complex Trajectory)"
else:
    policy_status = f"Early Handover at Step {chosen_step} (Certified Stable, Savings Achieved!)"

nfe_used = get_nfe(chosen_step)
nfe_saved_pct = (1.0 - nfe_used / 21.0) * 100.0

print(f"\n[DECISION]: Selected Handover Target: STEP {chosen_step}")
print(f"  Status          : {policy_status}")
print(f"  Forward Passes  : {nfe_used} NFE (Baseline Step 8 uses 21 NFE)")
print(f"  Compute Savings : {nfe_saved_pct:.1f}% Reduction in Neural Integration")

--- 
### Step 5: Student Consistency Completion ($j=1, \tau \in \{0.0761, 0.2929, 0.6173\}$)

The student consistency model receives the teacher intermediate endpoint $\hat{x}_1(t_{\hat{s}})$ and executes 3 consistency refinement passes along the swayed restart grid.

In [ ]:
start_student_time = time.time()
handover_endpoint = endpoints[chosen_step]

print("=" * 75)
print(f"STEP 5: STUDENT CONSISTENCY REFINEMENT (From Step {chosen_step} Endpoint):")

with torch.inference_mode():
    mel, synth_audio = run_student_completion(
        tts=tts,
        cond=cond_tensor,
        text=text_ids,
        teacher_endpoint=handover_endpoint,
        prompt_len=prompt_len,
        input_seed=BASE_SEED,
    )

student_elapsed = time.time() - start_student_time
total_elapsed = teacher_elapsed + student_elapsed
audio_len_sec = len(synth_audio) / 24000.0
rtf = total_elapsed / audio_len_sec

print(f"\n[OK] Waveform Synthesis Complete!")
print(f"  Output Waveform Length : {len(synth_audio):,} samples ({audio_len_sec:.2f} seconds)")
print(f"  Total Synthesis Time   : {total_elapsed:.2f} seconds (RTF: {rtf:.3f}x)")
print("=" * 75)

--- 
### Step 6: Interactive Listening, Spectrogram Visualization & Final Scorecard

Listen to the synthesized speech below and examine the spectrogram representation.

In [ ]:
# ------------------------------------------------------------------
# Interactive Audio Players
# ------------------------------------------------------------------
print("[AUDIO] 1. Reference Audio (Voice Clone Prompt):")
ipd.display(ipd.Audio(wav_np, rate=24000))

print(f"\n[AUDIO] 2. Synthesized Speech (Proposed Adaptive Policy: Step {chosen_step}):")
ipd.display(ipd.Audio(synth_audio, rate=24000))

# ------------------------------------------------------------------
# Spectrogram Visualization
# ------------------------------------------------------------------
mel_np = mel[0].detach().cpu().numpy()
plt.figure(figsize=(12, 4), dpi=120)
plt.imshow(mel_np, aspect='auto', origin='lower', cmap='viridis')
plt.axvline(x=prompt_len, color='red', linestyle='--', linewidth=2, label=f'Prompt / Synthesis Boundary ({prompt_len} frames)')
plt.title(f"DMOSpeech 2 Mel-Spectrogram — Proposed Adaptive Handover at Step {chosen_step} ({nfe_saved_pct:.1f}% Compute Saved)", fontsize=12, fontweight='bold')
plt.xlabel("Time (Frames)", fontsize=10)
plt.ylabel("Mel Filter Channels (1-100)", fontsize=10)
plt.colorbar(label="Log Mel Energy (dB)")
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# Executive Scorecard
# ------------------------------------------------------------------
print("\n" + "=" * 75)
print("CHAPTER 3 EXECUTIVE SCORECARD:")
print("=" * 75)
print(f"  Prompt SNR            : {prompt_snr:.2f} dB (Gate: >= {SAFETY_GATE_THETA:.1f} dB)")
print(f"  Safety Gate Fired     : {'YES (Routing to Step 8)' if gate_tripped else 'NO (Acoustically Clean)'}")
print(f"  Handover Step Selected: STEP {chosen_step}")
print(f"  Forward Passes (NFE)  : {nfe_used} NFE (Baseline = 21 NFE)")
print(f"  Compute Reduction     : {nfe_saved_pct:.1f}%")
print(f"  Total Elapsed Runtime : {total_elapsed:.2f}s (Real-Time Factor: {rtf:.3f}x)")
print(f"  Acoustic Integrity    : Certified R <= 0.05 under Learn-Then-Test bound")
print("=" * 75)